# Phase 2: Inventory Optimization (Genetic Algorithm)

Use GA to find optimal restocking quantities minimizing cost.

## Import Libraries

In [32]:
import pickle
import pandas as pd
import numpy as np
import random

## Load Model, Encoder, and Data

In [33]:
# Load model and encoder
with open('demand_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)

# Load data for categories
df = pd.read_csv('inventory_data.csv')
categories = df['ProductCategory'].unique()

print("Model, encoder, and data loaded.")

Model, encoder, and data loaded.


## Define GA Parameters

Set the rules for the evolutionary algorithm:
- Population Size: 30
- Run Limit: 100 Generations
- Mutation Rate: 10%

Define the financial penalties for inventory decisions:
- Holding Cost: 0.05 (penalty for storing excess stock)
- Shortage Cost: 1.5 (penalty for missed sales)


In [34]:
# GA Parameters
POPULATION_SIZE = 30
GENERATIONS = 100
MUTATION_RATE = 0.1

# Costs
HOLDING_COST = 0.05  # per unit per day
SHORTAGE_COST = 1.5  # per unit

# Assume average days for simulation (or use from data)
avg_days = df['DaysUntilExpiry'].mean()

print(f"GA Params: Pop={POPULATION_SIZE}, Gen={GENERATIONS}, Mut={MUTATION_RATE}")
print(f"Costs: Holding={HOLDING_COST}, Shortage={SHORTAGE_COST}")

GA Params: Pop=30, Gen=100, Mut=0.1
Costs: Holding=0.05, Shortage=1.5


## Define GA Functions

In [35]:
def initialize_population():
    population = []
    for _ in range(POPULATION_SIZE):
        individual = [random.uniform(0, 1000) for _ in categories]
        population.append(individual)
    return population

def predict_demand(quantity, days, category):
    input_df = pd.DataFrame([[category]], columns=['ProductCategory'])
    encoded = encoder.transform(input_df)
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['ProductCategory']))
    features = pd.concat([pd.DataFrame([[quantity, days]], columns=['QuantityInStock', 'DaysUntilExpiry']), encoded_df], axis=1)
    return model.predict(features)[0]

def fitness(individual):
    total_cost = 0
    for i, cat in enumerate(categories):
        restock = individual[i]
        demand = predict_demand(restock, avg_days, cat)
        excess = max(0, restock - demand)
        shortage = max(0, demand - restock)
        cost = HOLDING_COST * excess + SHORTAGE_COST * shortage
        total_cost += cost
    return total_cost

def crossover(parent1, parent2):
    child = []
    for i in range(len(parent1)):
        if random.random() < 0.5:
            child.append(parent1[i])
        else:
            child.append(parent2[i])
    return child

def mutate(individual):
    for i in range(len(individual)):
        if random.random() < MUTATION_RATE:
            individual[i] += random.uniform(-100, 100)
            individual[i] = max(0, individual[i])  # non-negative
    return individual

print("GA functions defined.")

GA functions defined.


## Run the Evolution Loop

In [36]:
# Initialize population
population = initialize_population()

for gen in range(GENERATIONS):
    # Evaluate fitness
    fitness_scores = [(ind, fitness(ind)) for ind in population]
    fitness_scores.sort(key=lambda x: x[1])  # sort by cost ascending
    
    # Select top half
    selected = [ind for ind, _ in fitness_scores[:POPULATION_SIZE//2]]
    
    # Crossover to create new population
    new_population = selected.copy()
    while len(new_population) < POPULATION_SIZE:
        parent1, parent2 = random.sample(selected, 2)
        child = crossover(parent1, parent2)
        child = mutate(child)
        new_population.append(child)
    
    population = new_population
    
    if gen % 10 == 0:
        best_cost = fitness_scores[0][1]
        print(f"Generation {gen}: Best cost = {best_cost:.2f}")

# Final best
best_individual = fitness_scores[0][0]
best_cost = fitness_scores[0][1]

print("\nOptimal restocking quantities:")
for i, cat in enumerate(categories):
    print(f"{cat}: {round(best_individual[i])}")
print(f"Total cost: {best_cost:.2f}")

Generation 0: Best cost = 46.79
Generation 10: Best cost = 13.11
Generation 20: Best cost = 5.20
Generation 30: Best cost = 3.09
Generation 40: Best cost = 2.71
Generation 50: Best cost = 1.89
Generation 60: Best cost = 1.67
Generation 70: Best cost = 1.21
Generation 80: Best cost = 0.91
Generation 90: Best cost = 0.91

Optimal restocking quantities:
Equipment: 157
Medicine: 210
Supplies: 197
Produce: 172
Dairy: 182
Bakery: 177
Total cost: 0.91
